# 03 — Publish to ArcGIS Online

This notebook publishes the processed site-year GeoDataFrame as a
Hosted Feature Layer on ArcGIS Online.

**Prerequisites:**
- Run `02_pipeline_demo.ipynb` first to generate the processed GeoJSON.
- Set environment variables `AGOL_USERNAME` and `AGOL_PASSWORD`
  (see README.md for details).

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s")

import geopandas as gpd
from pathlib import Path
from src import publish

## Load processed data

In [ ]:
SOURCE_ID = "hawaii"
data_path = Path("..") / "data" / "processed" / f"{SOURCE_ID}_site_summary.geojson"

gdf = gpd.read_file(data_path)
print(f"Loaded {len(gdf):,} features from {data_path.name}")
gdf.head(3)

## Publish to ArcGIS Online

In [ ]:
url = publish.publish_site_summary(
    gdf=gdf,
    source_id=SOURCE_ID,
    overwrite=True,
)
print(f"\nPublished Feature Layer URL:\n  {url}")

## Verify: query the published layer

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

fl = FeatureLayer(url + "/0")
count = fl.query(return_count_only=True)
print(f"Feature count on AGOL: {count}")
print(f"Local feature count:   {len(gdf)}")
assert count == len(gdf), "Mismatch between published and local counts!"
print("Counts match.")

## Display inline map

In [ ]:
import os
from arcgis.gis import GIS

gis = GIS(
    os.environ.get("AGOL_URL", "https://www.arcgis.com"),
    os.environ.get("AGOL_USERNAME"),
    os.environ.get("AGOL_PASSWORD"),
)

m = gis.map("Hawaii")
m.zoom = 7
m.add_layer(FeatureLayer(url + "/0"))
m